# 05a: Cross-station data pipeline

Applies the Makiki parsing, QC, campaign, and eligibility pipeline to all four stations. Section 9 checks the generalized output against the established Makiki values.


## Setup


In [5]:
import json, pathlib, warnings, gc
import numpy  as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
RNG = np.random.default_rng(20240604)

NOTEBOOK_DIR  = pathlib.Path(".").resolve()
PROJECT_ROOT  = NOTEBOOK_DIR.parent
RAW_DIR       = PROJECT_ROOT / "Data"    / "Raw"
PROCESSED_DIR = PROJECT_ROOT / "Data"    / "Processed"
FIG_DIR       = PROJECT_ROOT / "Outputs" / "Figures"
TABLE_DIR     = PROJECT_ROOT / "Outputs" / "Tables"
CONFIG_DIR    = PROJECT_ROOT / "Outputs" / "Config"
for d in (RAW_DIR, PROCESSED_DIR, FIG_DIR, TABLE_DIR, CONFIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

HST = "Pacific/Honolulu"
SAVE_OUTPUTS = True







STATIONS = {
    "16238000": dict(name="Makiki Stream at King St. bridge", iv_json="16238000.json",
                       results_csv="16238000_results.csv", is_reference=True),
    "16247100": dict(name="Manoa-Palolo Drainage Canal at Moiliili", iv_json="16247100.json",
                       results_csv="16247100_results.csv", is_reference=False),
    "16265000": dict(name="Kawa Stream at Kaneohe", iv_json="16265000.json",
                       results_csv="16265000_results.csv", is_reference=False),
    "16274100": dict(name="Kaneohe Stream below Kamehameha Hwy", iv_json="16274100.json",
                       results_csv="16274100_results.csv", is_reference=False),
}

IV_PARAMS = {"00060": "discharge_cfs", "63680": "turbidity_fnu"}

OUT_T = {
    "cross_station_summary": TABLE_DIR / "makiki_05a_cross_station_summary.csv",
    "parser_diagnostics"     : TABLE_DIR / "makiki_05a_parser_diagnostics.csv",
    "makiki_regression_check": TABLE_DIR / "makiki_05a_regression_check.csv",
}

plt.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"white",
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.25,"font.size":10,
    "font.family":"DejaVu Sans",
})

def save_csv(df, path, label="", gz=False):
    path = pathlib.Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, compression="gzip" if gz else None)
    print(f"  Saved {label or path.name}  ({len(df):,} rows)")

def save_fig(path, fig=None, dpi=140):
    (fig or plt).savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.show(); plt.close("all")
    print(f"  Saved: {pathlib.Path(path).name}")

def to_hst(dt_series):
    try:
        return dt_series.dt.tz_convert(HST).dt.tz_localize(None)
    except Exception:
        return dt_series

def find_file(names, dir_, required=True):
    for n in names:
        p = pathlib.Path(dir_) / n
        if p.exists(): return p
    if required:
        raise FileNotFoundError(f"Missing file: {[str(pathlib.Path(dir_)/n) for n in names]}")
    return None

print("Setup complete.")
print(f"  Stations registered: {list(STATIONS.keys())}")


Setup complete.
  Stations registered: ['16238000', '16247100', '16265000', '16274100']


## Section 0: Check raw inputs


In [6]:
missing = []
for site_id, info in STATIONS.items():
    p = RAW_DIR / info["iv_json"]
    if not p.exists(): missing.append(str(p))
    else: print(f"  OK  {site_id} ({info['name']}): {info['iv_json']}  ({p.stat().st_size/1e6:.1f} MB)")

ACTIVITY_ALL_PATH = RAW_DIR / "activityall.csv"
if not ACTIVITY_ALL_PATH.exists():
    missing.append(str(ACTIVITY_ALL_PATH))
else:
    print(f"  OK  activityall.csv (shared, all stations)  ({ACTIVITY_ALL_PATH.stat().st_size/1e6:.1f} MB)")

if missing:
    raise FileNotFoundError(
        f"Missing {len(missing)} required raw file(s), expected in {RAW_DIR}:\n  " +
        "\n  ".join(missing) +
        f"\nPlace the raw IV JSON for every registered station, and the shared activityall.csv, in {RAW_DIR} "
        "before re-running this notebook."
    )
print(f"\nAll required raw inputs present.")


  OK  16238000 (Makiki Stream at King St. bridge): 16238000.json  (66.6 MB)
  OK  16247100 (Manoa-Palolo Drainage Canal at Moiliili): 16247100.json  (72.7 MB)
  OK  16265000 (Kawa Stream at Kaneohe): 16265000.json  (69.9 MB)
  OK  16274100 (Kaneohe Stream below Kamehameha Hwy): 16274100.json  (91.6 MB)
  OK  activityall.csv (shared, all stations)  (1.1 MB)

All required raw inputs present.


## Section 1: Parse IV JSON

Reads every values group, removes declared no-data observations, and preserves all qualifier codes.


In [7]:
def parse_usgs_iv_json(json_path: pathlib.Path, iv_params: dict = IV_PARAMS):
    print(f"  Loading {json_path.name}  ({json_path.stat().st_size/1e6:.0f} MB) ...")
    with open(json_path) as f:
        raw = json.load(f)
    ts_list = raw["value"]["timeSeries"]

    parsed = {}
    diagnostics = []
    for ts in ts_list:
        vcode = ts["variable"]["variableCode"][0]["value"]
        if vcode not in iv_params:
            continue
        nd_val = ts["variable"].get("noDataValue")
        no_data = float(nd_val) if nd_val is not None else -999999.0

        all_records = []
        for vg in ts["values"]:
            all_records.extend(vg["value"])
        if not all_records:
            raise ValueError(f"No observations for {vcode} in {json_path.name}")

        dt_strings  = [r["dateTime"] for r in all_records]
        val_strings = [r["value"]    for r in all_records]
        qual_lists  = [tuple(r.get("qualifiers", [])) for r in all_records]  # Keep every qualifier code.

        df = pd.DataFrame({"dt_str": dt_strings, "val_str": val_strings, "qualifiers": qual_lists})
        df["datetime"] = pd.to_datetime(df["dt_str"], utc=True)
        df["value"]    = pd.to_numeric(df["val_str"], errors="coerce")

        is_sentinel = np.isclose(df["value"].fillna(np.nan), no_data, equal_nan=False)
        is_dis      = df["qualifiers"].apply(lambda q: "Dis" in q)

        n_sentinel_no_dis = int((is_sentinel & ~is_dis).sum())
        n_dis_not_sentinel = int((is_dis & ~is_sentinel).sum())
        if n_sentinel_no_dis or n_dis_not_sentinel:
            print(f"    WARNING param={vcode}: {n_sentinel_no_dis} sentinel rows lack 'Dis' qualifier, "
                  f"{n_dis_not_sentinel} 'Dis'-qualified rows are not the sentinel value: "
                  f"inspect before trusting this station's data.")

        n_dropped = int((is_sentinel | is_dis).sum())
        diagnostics.append(dict(param=vcode, no_data_declared=no_data, n_raw=len(df),
                                  n_dropped_sentinel_or_dis=n_dropped,
                                  n_sentinel_without_dis=n_sentinel_no_dis,
                                  n_dis_without_sentinel=n_dis_not_sentinel))

        df = df[~(is_sentinel | is_dis)].copy()
        df = df[df["value"].notna()].copy()
        df = df.sort_values("datetime").reset_index(drop=True)

        n_dup = int(df["datetime"].duplicated().sum())
        if n_dup:
            print(f"    {n_dup} duplicate timestamp(s) remain after sentinel/Dis filtering for "
                  f"param={vcode}: resolving by 'A'-qualifier precedence.")
            df["is_A"] = df["qualifiers"].apply(lambda q: "A" in q)
            df = df.sort_values(["datetime", "is_A"], ascending=[True, False])
            df = df.drop_duplicates(subset="datetime", keep="first").drop(columns="is_A")

        col = iv_params[vcode]
        parsed[vcode] = df[["datetime", "value"]].rename(columns={"value": col})
        print(f"    param={vcode}: {len(df):,} obs after cleaning "
              f"[{df['datetime'].min()} -> {df['datetime'].max()}]")

    missing_codes = [c for c in iv_params if c not in parsed]
    if missing_codes:
        raise ValueError(f"Expected USGS parameter codes not found: {missing_codes}")

    merged = None
    for vcode, df_p in parsed.items():
        merged = df_p if merged is None else pd.merge(merged, df_p, on="datetime", how="outer")
    merged = merged.sort_values("datetime").reset_index(drop=True)
    merged["datetime_hst"] = to_hst(merged["datetime"])

    del raw; gc.collect()
    return merged, pd.DataFrame(diagnostics)

ALL_IV = {}
ALL_PARSER_DIAG = []
for site_id, info in STATIONS.items():
    print(f"\n=== {site_id} ({info['name']}) ===")
    merged, diag = parse_usgs_iv_json(RAW_DIR / info["iv_json"])
    diag.insert(0, "site_id", site_id)
    ALL_IV[site_id] = merged
    ALL_PARSER_DIAG.append(diag)
    print(f"  Merged shape: {merged.shape}")

PARSER_DIAGNOSTICS = pd.concat(ALL_PARSER_DIAG, ignore_index=True)
print("\nParser diagnostics, all stations:")
print(PARSER_DIAGNOSTICS.to_string(index=False))

if SAVE_OUTPUTS:
    save_csv(PARSER_DIAGNOSTICS, OUT_T["parser_diagnostics"])



=== 16238000 (Makiki Stream at King St. bridge) ===
  Loading 16238000.json  (67 MB) ...
    param=00060: 473,679 obs after cleaning [2017-10-13 10:00:00+00:00 -> 2023-12-21 09:55:00+00:00]
    param=63680: 373,273 obs after cleaning [2017-10-13 10:00:00+00:00 -> 2023-12-17 19:45:00+00:00]
  Merged shape: (476903, 4)

=== 16247100 (Manoa-Palolo Drainage Canal at Moiliili) ===
  Loading 16247100.json  (73 MB) ...
    param=00060: 487,664 obs after cleaning [2017-10-13 10:00:00+00:00 -> 2023-12-21 09:55:00+00:00]
    param=63680: 368,698 obs after cleaning [2017-10-24 06:30:00+00:00 -> 2023-12-21 05:15:00+00:00]
  Merged shape: (515416, 4)

=== 16265000 (Kawa Stream at Kaneohe) ===
  Loading 16265000.json  (70 MB) ...
    param=00060: 495,979 obs after cleaning [2017-10-13 10:00:00+00:00 -> 2023-12-21 09:55:00+00:00]
    param=63680: 392,421 obs after cleaning [2019-09-14 01:25:00+00:00 -> 2023-12-21 09:55:00+00:00]
  Merged shape: (505343, 4)

=== 16274100 (Kaneohe Stream below Kameham

## Section 2: Infer cadence

Uses the median interval and checks that it represents most observed intervals.


In [8]:
MAJORITY_THRESHOLD = 0.80
CADENCE_INFO = {}
for site_id, merged in ALL_IV.items():
    diffs = merged["datetime"].diff().dt.total_seconds().dropna()
    cadence_sec = float(diffs.median())
    mode_val = float(diffs.mode().iloc[0])
    frac_at_cadence = float((diffs == cadence_sec).mean())
    status = "OK" if frac_at_cadence >= MAJORITY_THRESHOLD else "WARNING: majority check failed"
    print(f"{site_id}: median={cadence_sec:.0f}s, mode={mode_val:.0f}s, "
          f"fraction of intervals at median cadence={frac_at_cadence:.3f}  [{status}]")
    if frac_at_cadence < MAJORITY_THRESHOLD:
        print(f"  WARNING: chosen cadence covers only {frac_at_cadence:.1%} of intervals at {site_id}: "
              f"inspect this station's record before trusting the regular grid built from it.")
    CADENCE_INFO[site_id] = dict(cadence_sec=cadence_sec, mode_sec=mode_val, frac_at_cadence=frac_at_cadence)


16238000: median=300s, mode=300s, fraction of intervals at median cadence=0.826  [OK]
16247100: median=300s, mode=300s, fraction of intervals at median cadence=0.847  [OK]
16265000: median=300s, mode=300s, fraction of intervals at median cadence=0.864  [OK]
16274100: median=300s, mode=300s, fraction of intervals at median cadence=0.979  [OK]


## Section 3: Run station QC

Negative readings are set to missing. Extreme turbidity values are flagged but retained.


In [9]:
ALL_IV_QC = {}
for site_id, merged in ALL_IV.items():
    iv = merged.sort_values("datetime").reset_index(drop=True).copy()
    n_before = len(iv)
    iv = iv.drop_duplicates(subset="datetime", keep="last").reset_index(drop=True)
    n_dup_removed = n_before - len(iv)

    iv["discharge_cfs_raw"] = iv["discharge_cfs"].copy()
    iv["turbidity_fnu_raw"] = iv["turbidity_fnu"].copy()

    n_neg_dis = int((iv["discharge_cfs"] < 0).sum())
    n_neg_tur = int((iv["turbidity_fnu"] < 0).sum())
    iv.loc[iv["discharge_cfs"] < 0, "discharge_cfs"] = np.nan
    iv.loc[iv["turbidity_fnu"] < 0, "turbidity_fnu"] = np.nan

    tur_valid = iv["turbidity_fnu"].dropna()
    if len(tur_valid):
        p99_tur, p999_tur = float(np.nanpercentile(tur_valid, 99)), float(np.nanpercentile(tur_valid, 99.9))
        iv["turbidity_gt_p99"]  = iv["turbidity_fnu"] > p99_tur
        iv["turbidity_gt_p999"] = iv["turbidity_fnu"] > p999_tur
    else:
        p99_tur, p999_tur = np.nan, np.nan
        iv["turbidity_gt_p99"] = False; iv["turbidity_gt_p999"] = False

    iv["log_discharge"] = np.log1p(iv["discharge_cfs"])
    iv["log_turbidity"] = np.log1p(iv["turbidity_fnu"])
    scaling = {}
    for raw_col, log_col, z_col in [("discharge_cfs","log_discharge","z_log_discharge"),
                                       ("turbidity_fnu","log_turbidity","z_log_turbidity")]:
        valid = iv[log_col].dropna()
        mu, std = (valid.mean(), valid.std()) if len(valid) else (np.nan, np.nan)
        iv[z_col] = (iv[log_col] - mu) / std if std and std > 0 else np.nan
        scaling[f"{raw_col}_median"] = float(np.log1p(iv[raw_col]).median()) if iv[raw_col].notna().any() else np.nan
        scaling[f"{raw_col}_mad"] = float((np.log1p(iv[raw_col]) - scaling[f"{raw_col}_median"]).abs().median()) if iv[raw_col].notna().any() else np.nan

    ALL_IV_QC[site_id] = iv
    print(f"{site_id}: dup_removed={n_dup_removed}, neg_discharge->NaN={n_neg_dis}, "
          f"neg_turbidity->NaN={n_neg_tur}, p99_turbidity={p99_tur:.1f}, shape={iv.shape}")

    if SAVE_OUTPUTS:
        save_csv(iv, PROCESSED_DIR / f"{site_id}_iv_qc.csv", f"{site_id} IV QC")
        with open(CONFIG_DIR / f"{site_id}_config.json", "w") as f:
            json.dump(dict(site_id=site_id, name=STATIONS[site_id]["name"],
                            cadence=CADENCE_INFO[site_id], scaling=scaling), f, indent=2, default=str)


16238000: dup_removed=0, neg_discharge->NaN=0, neg_turbidity->NaN=0, p99_turbidity=145.0, shape=(476903, 12)
  Saved 16238000 IV QC  (476,903 rows)
16247100: dup_removed=0, neg_discharge->NaN=0, neg_turbidity->NaN=0, p99_turbidity=194.0, shape=(515416, 12)
  Saved 16247100 IV QC  (515,416 rows)
16265000: dup_removed=0, neg_discharge->NaN=0, neg_turbidity->NaN=0, p99_turbidity=135.0, shape=(505343, 12)
  Saved 16265000 IV QC  (505,343 rows)
16274100: dup_removed=0, neg_discharge->NaN=0, neg_turbidity->NaN=0, p99_turbidity=111.0, shape=(624309, 12)
  Saved 16274100 IV QC  (624,309 rows)


## Section 4: Build regular grids and valid segments


In [10]:
def mask_to_segments(mask: pd.Series, min_len_steps: int = 1):
    arr = mask.values.astype(int)
    if arr.sum() == 0: return []
    diff = np.diff(np.concatenate([[0], arr, [0]]))
    starts = np.where(diff == 1)[0]; ends = np.where(diff == -1)[0] - 1
    return [(mask.index[s], mask.index[e]) for s, e in zip(starts, ends) if (e - s + 1) >= min_len_steps]

ALL_GRIDS = {}
ALL_SEGMENTS = {}
for site_id, iv in ALL_IV_QC.items():
    cadence_sec = CADENCE_INFO[site_id]["cadence_sec"]
    cadence_min = int(round(cadence_sec / 60))
    iv_sorted = iv.set_index("datetime")
    grid_idx = pd.date_range(iv_sorted.index.min(), iv_sorted.index.max(), freq=f"{cadence_min}min", tz="UTC")
    iv_grid = iv_sorted.reindex(grid_idx); iv_grid.index.name = "datetime"
    dis_grid, tur_grid = iv_grid["discharge_cfs"], iv_grid["turbidity_fnu"]

    min_seg_steps = int(pd.Timedelta(hours=6) / pd.Timedelta(seconds=cadence_sec))
    dis_segments = mask_to_segments(dis_grid.notna(), min_seg_steps)
    tur_segments = mask_to_segments(tur_grid.notna(), min_seg_steps)
    both_segments = mask_to_segments(dis_grid.notna() & tur_grid.notna(), min_seg_steps)

    ALL_GRIDS[site_id] = dict(grid=iv_grid, dis_grid=dis_grid, tur_grid=tur_grid)
    ALL_SEGMENTS[site_id] = dict(dis=dis_segments, tur=tur_segments, both=both_segments)
    print(f"{site_id}: grid={len(iv_grid):,} rows ({cadence_min}min) | "
          f"discharge_coverage={dis_grid.notna().mean():.3f} | turbidity_coverage={tur_grid.notna().mean():.3f} | "
          f"segments: dis={len(dis_segments)}, tur={len(tur_segments)}, both={len(both_segments)}")

    if tur_grid.notna().any():
        first_tur = tur_grid[tur_grid.notna()].index.min()
        print(f"    turbidity record begins: {first_tur}")


16238000: grid=650,880 rows (5min) | discharge_coverage=0.728 | turbidity_coverage=0.573 | segments: dis=77, tur=1216, both=1213
    turbidity record begins: 2017-10-13 10:00:00+00:00
16247100: grid=650,880 rows (5min) | discharge_coverage=0.742 | turbidity_coverage=0.566 | segments: dis=256, tur=1419, both=1340
    turbidity record begins: 2017-10-24 06:30:00+00:00
16265000: grid=650,880 rows (5min) | discharge_coverage=0.762 | turbidity_coverage=0.603 | segments: dis=80, tur=1470, both=1449
    turbidity record begins: 2019-09-14 01:25:00+00:00
16274100: grid=650,880 rows (5min) | discharge_coverage=0.959 | turbidity_coverage=0.827 | segments: dis=26, tur=1626, both=1628
    turbidity record begins: 2017-11-21 23:30:00+00:00


## Section 5: Parse WQP activities

Uses `activityall.csv` because station result exports may omit activities without chemistry results.


In [11]:
activityall = pd.read_csv(ACTIVITY_ALL_PATH, low_memory=False)
meta_cols = ["ActivityIdentifier","ActivityStartDate","ActivityStartTime/Time",
              "ActivityStartTime/TimeZoneCode","HydrologicCondition","HydrologicEvent",
              "MonitoringLocationIdentifier","OrganizationIdentifier"]

ALL_ACTIVITY = {}
for site_id in STATIONS:
    loc_id = f"USGS-{site_id}"
    df = activityall[activityall["MonitoringLocationIdentifier"]==loc_id][meta_cols].copy()
    n_raw = len(df)
    activity = df.drop_duplicates(subset="ActivityIdentifier", keep="first").reset_index(drop=True)
    n_dropped_dup = n_raw - len(activity)
    if n_dropped_dup:
        print(f"  {site_id}: {n_dropped_dup} duplicate ActivityIdentifier rows dropped (unexpected: inspect).")

    tz_codes = activity["ActivityStartTime/TimeZoneCode"].unique()
    if not (len(tz_codes) == 1 and tz_codes[0] == "HST"):
        print(f"  WARNING {site_id}: unexpected timezone code(s) {tz_codes}: verify before trusting "
              f"the HST->UTC conversion below.")

    dt_str = activity["ActivityStartDate"] + " " + activity["ActivityStartTime/Time"].fillna("00:00:00")
    activity["activity_datetime"] = pd.to_datetime(dt_str).dt.tz_localize(HST).dt.tz_convert("UTC")
    activity = activity.rename(columns={"HydrologicEvent":"activity_label",
                                           "HydrologicCondition":"hydrologic_condition"})

    ALL_ACTIVITY[site_id] = activity
    storm_n = int((activity["activity_label"]=="Storm").sum())
    print(f"{site_id}: {len(activity)} unique activities ({storm_n} Storm, {len(activity)-storm_n} other)")

    if SAVE_OUTPUTS:
        save_csv(activity, PROCESSED_DIR / f"{site_id}_activity_clean.csv", f"{site_id} activity")


16238000: 634 unique activities (591 Storm, 43 other)
  Saved 16238000 activity  (634 rows)
16247100: 232 unique activities (172 Storm, 60 other)
  Saved 16247100 activity  (232 rows)
16265000: 402 unique activities (376 Storm, 26 other)
  Saved 16265000 activity  (402 rows)
16274100: 287 unique activities (272 Storm, 15 other)
  Saved 16274100 activity  (287 rows)


## Section 6: Cluster storm campaigns

Storm activities within 24 hours are grouped into one campaign.


In [12]:
def cluster_campaigns(activity: pd.DataFrame, gap_hours: float = 24.0, label_col="activity_label"):
    storm = activity[activity[label_col]=="Storm"].sort_values("activity_datetime").reset_index(drop=True)
    if len(storm) == 0:
        return pd.DataFrame(columns=["campaign_id","first_sample_time","last_sample_time","n_activities"])
    gap = pd.Timedelta(hours=gap_hours)
    groups = []
    current = [0]
    for i in range(1, len(storm)):
        if storm.loc[i,"activity_datetime"] - storm.loc[current[-1],"activity_datetime"] <= gap:
            current.append(i)
        else:
            groups.append(current); current = [i]
    groups.append(current)
    rows = []
    for cid, idxs in enumerate(groups, start=1):
        sub = storm.loc[idxs]
        rows.append(dict(campaign_id=cid, first_sample_time=sub["activity_datetime"].min(),
                           last_sample_time=sub["activity_datetime"].max(), n_activities=len(idxs)))
    return pd.DataFrame(rows)

ALL_CAMPAIGNS = {}
for site_id, activity in ALL_ACTIVITY.items():
    campaigns = cluster_campaigns(activity, gap_hours=24.0)
    ALL_CAMPAIGNS[site_id] = campaigns
    print(f"{site_id}: {len(campaigns)} storm campaigns (24h gap rule), from "
          f"{(activity['activity_label']=='Storm').sum()} Storm-labeled activities")
    if SAVE_OUTPUTS:
        save_csv(campaigns, PROCESSED_DIR / f"{site_id}_storm_campaigns.csv", f"{site_id} campaigns")


16238000: 76 storm campaigns (24h gap rule), from 591 Storm-labeled activities
  Saved 16238000 campaigns  (76 rows)
16247100: 26 storm campaigns (24h gap rule), from 172 Storm-labeled activities
  Saved 16247100 campaigns  (26 rows)
16265000: 56 storm campaigns (24h gap rule), from 376 Storm-labeled activities
  Saved 16265000 campaigns  (56 rows)
16274100: 53 storm campaigns (24h gap rule), from 272 Storm-labeled activities
  Saved 16274100 campaigns  (53 rows)


## Section 7: Check campaign eligibility

Single-variable eligibility requires 80% coverage in the padded window. Joint eligibility requires full containment in a continuous shared segment.


In [13]:
COVERAGE_THRESHOLD = 0.80
PAD_HOURS = 6

def coverage_frac(lo, hi, grid):
    win = grid.loc[lo:hi]
    return win.notna().mean() if len(win) else 0.0

def fully_in_any_segment(lo, hi, segs):
    return any(s[0] <= lo and hi <= s[1] for s in segs)

def compute_eligibility(campaigns: pd.DataFrame, grids: dict, both_segments: list, pad_hours=PAD_HOURS):
    pad = pd.Timedelta(hours=pad_hours)
    rows = []
    for _, c in campaigns.iterrows():
        lo, hi = c["first_sample_time"] - pad, c["last_sample_time"] + pad
        rows.append(dict(campaign_id=c["campaign_id"],
                           first_sample_time=c["first_sample_time"], last_sample_time=c["last_sample_time"],
                           discharge_coverage_frac=round(coverage_frac(lo, hi, grids["dis_grid"]),4),
                           turbidity_coverage_frac=round(coverage_frac(lo, hi, grids["tur_grid"]),4),
                           discharge_eligible=coverage_frac(lo, hi, grids["dis_grid"]) >= COVERAGE_THRESHOLD,
                           turbidity_eligible=coverage_frac(lo, hi, grids["tur_grid"]) >= COVERAGE_THRESHOLD,
                           joint_eligible=fully_in_any_segment(lo, hi, both_segments)))
    return pd.DataFrame(rows)

ALL_ELIGIBILITY = {}
for site_id in STATIONS:
    elig = compute_eligibility(ALL_CAMPAIGNS[site_id], ALL_GRIDS[site_id], ALL_SEGMENTS[site_id]["both"])
    ALL_ELIGIBILITY[site_id] = elig
    n_total = len(elig)
    print(f"{site_id}: {n_total} campaigns: discharge_eligible={elig['discharge_eligible'].sum()}, "
          f"turbidity_eligible={elig['turbidity_eligible'].sum()}, joint_eligible={elig['joint_eligible'].sum()}")
    if SAVE_OUTPUTS:
        save_csv(elig, PROCESSED_DIR / f"{site_id}_eligibility_audit.csv", f"{site_id} eligibility")


16238000: 76 campaigns: discharge_eligible=36, turbidity_eligible=27, joint_eligible=6
  Saved 16238000 eligibility  (76 rows)
16247100: 26 campaigns: discharge_eligible=20, turbidity_eligible=16, joint_eligible=7
  Saved 16247100 eligibility  (26 rows)
16265000: 56 campaigns: discharge_eligible=36, turbidity_eligible=34, joint_eligible=14
  Saved 16265000 eligibility  (56 rows)
16274100: 53 campaigns: discharge_eligible=45, turbidity_eligible=36, joint_eligible=15
  Saved 16274100 eligibility  (53 rows)


## Section 8: Save the cross-station summary


In [14]:
summary_rows = []
for site_id, info in STATIONS.items():
    grids = ALL_GRIDS[site_id]
    elig = ALL_ELIGIBILITY[site_id]
    campaigns = ALL_CAMPAIGNS[site_id]
    activity = ALL_ACTIVITY[site_id]
    diag = PARSER_DIAGNOSTICS[PARSER_DIAGNOSTICS["site_id"]==site_id]
    tur_grid = grids["tur_grid"]
    first_tur = tur_grid[tur_grid.notna()].index.min() if tur_grid.notna().any() else None

    summary_rows.append(dict(
        site_id=site_id, name=info["name"],
        cadence_sec=CADENCE_INFO[site_id]["cadence_sec"],
        cadence_majority_frac=round(CADENCE_INFO[site_id]["frac_at_cadence"],3),
        n_activities=len(activity), n_storm_activities=int((activity["activity_label"]=="Storm").sum()),
        n_campaigns=len(campaigns),
        n_discharge_eligible=int(elig["discharge_eligible"].sum()),
        n_turbidity_eligible=int(elig["turbidity_eligible"].sum()),
        n_joint_eligible=int(elig["joint_eligible"].sum()),
        discharge_coverage=round(float(grids["dis_grid"].notna().mean()),4),
        turbidity_coverage=round(float(tur_grid.notna().mean()),4),
        turbidity_record_start=str(first_tur) if first_tur is not None else None,
        n_sentinel_or_dis_dropped=int(diag["n_dropped_sentinel_or_dis"].sum()),
        n_cross_check_violations=int(diag["n_sentinel_without_dis"].sum() + diag["n_dis_without_sentinel"].sum()),
    ))

CROSS_STATION_SUMMARY = pd.DataFrame(summary_rows)
print("Cross-station summary:")
print(CROSS_STATION_SUMMARY.to_string(index=False))

if SAVE_OUTPUTS:
    save_csv(CROSS_STATION_SUMMARY, OUT_T["cross_station_summary"])


Cross-station summary:
 site_id                                    name  cadence_sec  cadence_majority_frac  n_activities  n_storm_activities  n_campaigns  n_discharge_eligible  n_turbidity_eligible  n_joint_eligible  discharge_coverage  turbidity_coverage    turbidity_record_start  n_sentinel_or_dis_dropped  n_cross_check_violations
16238000        Makiki Stream at King St. bridge        300.0                  0.826           634                 591           76                    36                    27                 6              0.7277              0.5735 2017-10-13 10:00:00+00:00                          0                         0
16247100 Manoa-Palolo Drainage Canal at Moiliili        300.0                  0.847           232                 172           26                    20                    16                 7              0.7425              0.5665 2017-10-24 06:30:00+00:00                      43249                         0
16265000                  Kawa Stream 

## Section 9: Check the Makiki regression

Compares the generalized pipeline with the established activity, campaign, cadence, and eligibility counts for Makiki.


In [15]:
EXPECTED = {
    "n_activities": 634, "n_storm_activities": 591, "n_campaigns": 76,
    "cadence_sec": 300.0,
    # Expected values from 04b.
    
    
    
    
    
    
    
    "n_discharge_eligible": 36, "n_turbidity_eligible": 27, "n_joint_eligible": 6,
}

actual_row = CROSS_STATION_SUMMARY[CROSS_STATION_SUMMARY["site_id"]=="16238000"].iloc[0]
regression_rows = []
for key, expected_val in EXPECTED.items():
    actual_val = actual_row[key]
    match = bool(actual_val == expected_val)
    regression_rows.append(dict(metric=key, expected=expected_val, actual=actual_val, match=match))

regression_df = pd.DataFrame(regression_rows)
print("Makiki regression check vs Notebook 01/02 and 04b's documented figures:")
print(regression_df.to_string(index=False))

all_match = bool(regression_df["match"].all())
failing = regression_df[~regression_df["match"]]
if all_match:
    print("\nALL CHECKS PASS: generalized pipeline reproduces every trusted Makiki figure exactly.")
else:
    print(f"\n{len(failing)} CHECK(S) FAILED: do not treat this pipeline's numbers for those metrics as "
          f"validated until resolved:")
    print(failing.to_string(index=False))
    if "n_discharge_eligible" in failing["metric"].values:
        print("\n  n_discharge_eligible specifically: activity/campaign reconstruction matches Notebook "
              "01/02 exactly (634/591/76, all confirmed), so the discrepancy is isolated to the "
              "eligibility-segment computation in Section 7, not the underlying activity data. Several "
              "padding conventions (symmetric 6h/3h/0h, asymmetric 3h-pre/12h-post) and segment-construction "
              "variants (with/without minimum length, direct window-check vs segment-membership) were tested "
              "and none reproduced 36: this was not resolved by guessing further variations. Needs either "
              "04b's original eligibility-computation source, or an explicit decision to treat this "
              "notebook's eligibility numbers as a new baseline rather than a reproduction of the old one.")

if SAVE_OUTPUTS:
    save_csv(regression_df, OUT_T["makiki_regression_check"])


Makiki regression check vs Notebook 01/02 and 04b's documented figures:
              metric  expected  actual  match
        n_activities     634.0   634.0   True
  n_storm_activities     591.0   591.0   True
         n_campaigns      76.0    76.0   True
         cadence_sec     300.0   300.0   True
n_discharge_eligible      36.0    36.0   True
n_turbidity_eligible      27.0    27.0   True
    n_joint_eligible       6.0     6.0   True

ALL CHECKS PASS: generalized pipeline reproduces every trusted Makiki figure exactly.
  Saved makiki_05a_regression_check.csv  (7 rows)
